# GES YOLO26 — Google Colab eğitimi (free T4)

Cursor oturumu Colab GPU’suna bağlanamaz. Eğitim **bu notebook Colab’da açıkken** koşar; `best.pt` Drive’a yazılır, sonra yerel `models/best.pt` olarak indirilir.

**Hazırlık (yerelde, bir kez)**
1. Google Drive’da `GES/` klasörü oluştur.
2. `data/SOLAR PANEL DET.v1i.yolo26` (~569 MB) ve `data/ges_project.yaml` dosyasını oraya yükle.
3. Colab: **Runtime → Change runtime type → T4 GPU**.

**Free tier önerisi:** `SIZE = "m"` (YOLO26m). `l` riskli ama mümkün; `x` 12 saatlik oturumu aşabilir.

In [ ]:
SIZE = "m"          # n | s | m | l | x   — free T4 için m
EPOCHS = 70
IMGSZ = 640
BATCH = -1          # AutoBatch: T4 VRAM'in ~%60'ı
DRIVE_ROOT = "/content/drive/MyDrive/GES"

!nvidia-smi -L || true
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install -U ultralytics

import os, shutil
from pathlib import Path

work = Path("/content/ges")
work.mkdir(exist_ok=True)
os.chdir(work)

src = Path(DRIVE_ROOT)
assert src.exists(), f"Drive klasörü yok: {src}. GES/ altına veri setini yükle."

data_dst = work / "data"
data_dst.mkdir(exist_ok=True)
yaml_src = src / "ges_project.yaml"
ds_src = src / "SOLAR PANEL DET.v1i.yolo26"
zip_src = src / "SOLAR PANEL DET.v1i.yolo26.zip"

if yaml_src.exists():
    shutil.copy2(yaml_src, data_dst / "ges_project.yaml")
if ds_src.exists() and not (data_dst / "SOLAR PANEL DET.v1i.yolo26").exists():
    shutil.copytree(ds_src, data_dst / "SOLAR PANEL DET.v1i.yolo26")
elif zip_src.exists() and not (data_dst / "SOLAR PANEL DET.v1i.yolo26").exists():
    !unzip -q -o "{zip_src}" -d "{data_dst}"

print("data:", list(data_dst.iterdir()))

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import json, shutil

weights = {
    "n": "yolo26n.pt",
    "s": "yolo26s.pt",
    "m": "yolo26m.pt",
    "l": "yolo26l.pt",
    "x": "yolo26x.pt",
}[SIZE]

yaml = Path("data/ges_project.yaml")
assert yaml.exists(), "ges_project.yaml bulunamadı"

out_drive = Path(DRIVE_ROOT) / "models"
out_drive.mkdir(parents=True, exist_ok=True)
last_ckpt = out_drive / "ges_yolo26" / "weights" / "last.pt"

if last_ckpt.exists():
    print("Resume:", last_ckpt)
    model = YOLO(str(last_ckpt))
    results = model.train(resume=True)
else:
    model = YOLO(weights)
    results = model.train(
        data=str(yaml),
        epochs=EPOCHS,
        batch=BATCH,
        device=0,
        imgsz=IMGSZ,
        project=str(out_drive),
        name="ges_yolo26",
        exist_ok=True,
        save=True,
        save_period=5,
        patience=20,
        workers=2,
        degrees=5.0,
        hsv_h=0.0,
        hsv_s=0.2,
        hsv_v=0.2,
        fliplr=0.5,
        flipud=0.0,
        scale=0.1,
        translate=0.1,
        perspective=0.0,
        mosaic=0.5,
    )

best_src = out_drive / "ges_yolo26" / "weights" / "best.pt"
best_named = out_drive / f"best_yolo26{SIZE}.pt"
if best_src.exists():
    shutil.copy2(best_src, best_named)
    shutil.copy2(best_src, out_drive / "best.pt")
    print("Kaydedildi:", best_named)

metrics = {
    "architecture": f"yolo26{SIZE}",
    "mAP50": float(results.results_dict.get("metrics/mAP50(B)", 0)),
    "mAP50-95": float(results.results_dict.get("metrics/mAP50-95(B)", 0)),
    "precision": float(results.results_dict.get("metrics/precision(B)", 0)),
    "recall": float(results.results_dict.get("metrics/recall(B)", 0)),
    "epochs": EPOCHS,
}
(out_drive / "train_metrics.json").write_text(json.dumps(metrics, indent=2))
print(metrics)

Eğitim bitince yerelde:

```bash
cp ~/Drive/GES/models/best.pt models/best.pt
```

Oturum koparsa aynı notebook’u tekrar çalıştır; `last.pt` varsa `resume=True` ile devam eder.

| Boyut | T4 (16 GB) | ~70 epoch / 8550 görüntü | Free tier |
|---|---|---|---|
| s | rahat, batch 16–32 | ~2–4 saat | evet (zaten var) |
| **m** | batch −1 (~8–16) | ~4–7 saat | **önerilen** |
| l | batch −1 (~4–8) | ~7–11 saat | sınırda |
| x | batch 2–4 | ~12+ saat | hayır (oturum + kota) |